# धडा ०४ - टूल वापर डिझाइन पॅटर्न

या धड्यात तुम्ही Microsoft Agent Framework (Python) वापरून AI एजंटसाठी **टूल वापर** डिझाइन पॅटर्न शिकाल. आपण पुढील गोष्टींचा आढावा घेणार आहोत:

- `@tool` डेकोरेटर आणि टायप्ड पॅरामीटर्ससह फंक्शन टूल डिफाइन करणे
- मॉडेलला प्रत्येक टूल काय करते हे समजावे यासाठी टूल स्कीमा प्रदान करणे
- `approval_mode` सह टूलची अंमलबजावणी नियंत्रित करणे
- Pydantic मॉडेल्स आणि `response_format` द्वारे **संरचित आउटपुट** परत करणे

परिस्थिती ही एक **प्रवास बुकिंग एजंट** आहे जी ठिकाणे शोधू शकते, उपलब्धता तपासू शकते, आणि फ्लाइटची माहिती मिळवू शकते.


## सेटअप


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## @tool डेकोरेटरसह साधने व्याख्यित करणे

`@tool` डेकोरेटर साध्या Python फंक्शनला अशा साधनामध्ये रूपांतरित करतो जी एजंट कॉल करू शकतो.
महत्त्वाचे मुद्दे:

- **docstring** हे मॉडेलला दिसणारे साधनाचे वर्णन बनते.
- **टाइप अनोटेशन्स** (वर्णनांसह `Annotated` सह समाविष्ट) साधनाचा स्कीमा परिभाषित करतात.
- `approval_mode` नियंत्रित करते की प्रत्येक कॉल चालवण्यापूर्वी वापरकर्त्याला मंजुरी द्यावी लागेल की नाही.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## एकाधिक साधनांसह एजंट तयार करणे

सर्व तीन साधने क्लायंटकडे पास करा जेणेकरून मॉडेल वापरकर्त्याच्या प्रश्नाला उत्तर देण्यासाठी जे कोणतीही साधने आवश्यक असतील ती वापरू शकेल.


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = await provider.create_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## टूल्ससह संरचित आउटपुट

`response_format` ला Pydantic मॉडेलवर सेट केल्याने, एजंटला मुक्त स्वरूपाच्या मजकुराऐवजी व्यवस्थित टायप केलेला JSON ऑब्जेक्ट परत करण्यास भाग पडतो. जेव्हा खालील प्रक्रियेतील कोडला परिणाम प्रोग्रामेटिक पद्धतीने वापरायचा असतो तेव्हा हे उपयुक्त ठरते.


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = await provider.create_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## टूल मंजूरी नमुने

`@tool` वरील `approval_mode` पॅरामीटर टूल कॉल्सना अंमलबजावणीपूर्वी मानवी मंजुरी आवश्यक आहे का हे नियंत्रित करते:

| मोड | वर्तन |
|---|---|
| `"never_require"` | टूल आपोआप चालते — वापरकर्त्याच्या पुष्टीची गरज नाही. |
| `"always_require"` | प्रत्येक कॉलला चालण्याआधी वापरकर्त्याने मंजूर करणे आवश्यक आहे. |

ज्या टूल्सचे साइड-इफेक्ट्स असतात (उदा. फ्लाइट बुकिंग, क्रेडिट कार्ड चार्ज करणे) अशा टूल्ससाठी `"always_require"` वापरा जेणेकरून माणूस प्रक्रियेत राहील.


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## सारांश

या धड्यात तुम्ही कसे शिकणार आहात:

1. **टूल्स परिभाषित करा** `@tool` डेकोरेटर वापरून, ज्यामध्ये टाइप केलेले पॅरामीटर्स आणि डॉकस्ट्रींग्ज असतात जे टूल स्कीमा म्हणून कार्य करतात.
2. **एकाधिक टूल्स संयोजित करा** जेणेकरून एजंट ते अनुक्रमाने कॉल करू शकतो आणि गुंतागुंतीचे प्रश्नांची उत्तरे देऊ शकतो.
3. **संरचित आउटपुट परत करा** Pydantic मॉडेल `response_format` म्हणून पास करुन.
4. **टूल मंजुरी नियंत्रण करा** `approval_mode` वापरून जेणेकरून संवेदनशील ऑपरेशन्ससाठी मानवी परवानगी घेतली जाईल.

हे नमुने विश्वासार्ह, उत्पादनासाठी तयार एजंट तयार करण्याचा पाया तयार करतात जे बाह्य सिस्टम्सशी सुरक्षितपणे संवाद साधू शकतात.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:  
हा दस्तऐवज AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) वापरून अनुवादित केला आहे. आम्ही अचूकतेसाठी प्रयत्नशील असलो तरी, कृपया लक्षात घ्या की स्वयंचलित अनुवादांमध्ये चुका किंवा त्रुटी असू शकतात. मूळ दस्तऐवज त्याच्या स्थानिक भाषेत अधिकृत स्रोत मानला पाहिजे. महत्वाच्या माहितीच्या बाबतीत व्यावसायिक मानवी अनुवादाची शिफारस केली जाते. या अनुवादाच्या वापरामुळे उद्भवलेल्या कोणत्याही गैरसमजुतीसाठी किंवा चुकीच्या अर्थ लावणीनाही आम्ही जबाबदार नाही.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
